<a href="https://colab.research.google.com/github/estmichellartund-beep/IPYNB/blob/main/Tarea1_dataset_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto de Procesamiento Digital de Señales
## Análisis de señales temporales de matrícula en educación superior en Colombia (2016–2025)

**Integrantes:** _Mariana Castro, Yensy Martinez y Michelle Artunduaga_ &nbsp;|&nbsp;
**Fecha de descarga:** 10 de noviembre de 2026.

## 1. Código de construcción


In [ ]:
"""
Tarea 1 - Construcción del Dataset 1 (matrícula en educación superior, SNIES).
Fuente: SNIES > Estadísticas > Bases consolidadas > "Estudiantes matriculados AAAA".

Criterios (verificados contra cifras del portal/MEN en 2016, 2021 y 2025):
  * Territorio = 'Departamento de oferta del programa'.
  * Dato anual = semestre 1 de las IES distintas del SENA + semestre 2 del SENA
                 (nota metodológica del portal de consultas públicas).
  * oficial/privado = % sobre la matrícula TOTAL del territorio (todos los niveles).
  * tecnica/tecnologica/universitaria = niveles 'Formación técnica profesional',
    'Tecnológica(o)' y 'Universitaria(o)' (no incluye especializaciones, maestría ni doctorado).
Uso: poner los Excel (un archivo por año, con el año en el nombre) en ./snies_raw/ y ejecutar el script.
"""
import os, re, glob, unicodedata
import pandas as pd

def _sin_tildes(s):
    s = unicodedata.normalize("NFD", str(s))
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

def _norm(s):
    """Mayúsculas, sin tildes ni saltos de línea ni espacios repetidos."""
    return re.sub(r"\s+", " ", _sin_tildes(s).replace("\n", " ")).strip().upper()

# Nombres canónicos de los 33 territorios (clave normalizada -> nombre final)
CANON = ["Amazonas", "Antioquia", "Arauca", "Archipiélago de San Andrés, Providencia y Santa Catalina",
         "Atlántico", "Bogotá, D.C.", "Bolívar", "Boyacá", "Caldas", "Caquetá", "Casanare", "Cauca",
         "Cesar", "Chocó", "Cundinamarca", "Córdoba", "Guainía", "Guaviare", "Huila", "La Guajira",
         "Magdalena", "Meta", "Nariño", "Norte de Santander", "Putumayo", "Quindío", "Risaralda",
         "Santander", "Sucre", "Tolima", "Valle del Cauca", "Vaupés", "Vichada"]
CLAVE_A_CANON = {_norm(n): n for n in CANON}
# Variantes detectadas en los archivos (errores de codificación / truncamientos)
CLAVE_A_CANON.update({"NARINIO": "Nariño", "SAN ANDRES Y PROVI": CANON[3],
                      "BOGOTA D.C": "Bogotá, D.C.", "GUAJIRA": "La Guajira",
                      "SAN ANDRES Y PROVIDENCIA": CANON[3]})

def canon(x):
    """Devuelve el nombre canónico del territorio (tolera variantes, tildes y truncamientos)."""
    k = _norm(x)
    if k in CLAVE_A_CANON:
        return CLAVE_A_CANON[k]
    if k.startswith("BOGOTA"):
        return "Bogotá, D.C."
    if k.startswith("ARCHIPIELAGO") or k.startswith("SAN ANDRES"):
        return CANON[3]
    return None

NIVELES = {"FORMACION TECNICA PROFESIONAL": "tecnica",
           "TECNOLOGICA": "tecnologica", "TECNOLOGICO": "tecnologica",
           "UNIVERSITARIA": "universitaria", "UNIVERSITARIO": "universitaria"}

def leer_base(ruta):
    """Elige la hoja que contiene la columna 'Semestre', localiza el encabezado y normaliza columnas."""
    xl = pd.ExcelFile(ruta)
    for hoja in xl.sheet_names:
        cab = xl.parse(hoja, header=None, nrows=40)
        filas = [i for i in range(len(cab)) if cab.iloc[i].astype(str).map(_norm).eq("SEMESTRE").any()]
        if filas:
            df = xl.parse(hoja, header=filas[0])
            df.columns = [_norm(c) for c in df.columns]
            return df
    raise ValueError(f"No se encontró la columna 'Semestre' en {ruta}")

def agregar_anio(df, anio):
    col_m = next(c for c in df.columns if c.startswith("MATRICULADOS"))
    col_ies = next(c for c in df.columns if c.startswith("INSTITUCION DE EDUCACION SUPERIOR"))
    col_dep = next(c for c in df.columns if "DEPARTAMENTO DE OFERTA" in c)
    col_sec = next(c for c in df.columns if c in ("SECTOR IES",))
    col_niv = next(c for c in df.columns if c == "NIVEL DE FORMACION")

    df = df.copy()
    df[col_m] = pd.to_numeric(df[col_m], errors="coerce")
    df["SEMESTRE"] = pd.to_numeric(df["SEMESTRE"], errors="coerce")
    df = df[df[col_m].notna() & df["SEMESTRE"].isin([1, 2])]          # descarta filas de notas/pie

    sena = df[col_ies].map(_norm).str.contains("SENA")
    base = df[(~sena & (df["SEMESTRE"] == 1)) | (sena & (df["SEMESTRE"] == 2))].copy()

    base["territorio"] = base[col_dep].map(canon)
    sin_mapa = base[base["territorio"].isna()][col_dep].unique()
    if len(sin_mapa):
        print(f"[{anio}] AVISO: departamentos sin mapear (se excluyen): {sin_mapa}")
    base = base[base["territorio"].notna()]
    base["sector"] = base[col_sec].map(lambda x: "oficial" if _norm(x).startswith("OFICIAL") else "privado")
    base["nivel"] = base[col_niv].map(lambda x: NIVELES.get(_norm(x)))

    tot = base.groupby("territorio")[col_m].sum().rename("total")
    sec = base.pivot_table(index="territorio", columns="sector", values=col_m, aggfunc="sum", fill_value=0)
    niv = (base[base["nivel"].notna()]
           .pivot_table(index="territorio", columns="nivel", values=col_m, aggfunc="sum", fill_value=0))
    out = pd.concat([tot, sec, niv], axis=1).fillna(0)
    out["oficial_pct"] = (100 * out["oficial"] / out["total"]).round(1)
    out["privado_pct"] = (100 * out["privado"] / out["total"]).round(1)
    out = out.reset_index().rename(columns={"territorio": "departamento"})
    out["año"] = anio
    return out

def construir(carpeta="snies_raw"):
    piezas = []
    for ruta in sorted(glob.glob(f"{carpeta}/*.xlsx")):
        anio = int(re.search(r"(\d{4})", os.path.basename(ruta)).group(1))
        print("Procesando", ruta)
        piezas.append(agregar_anio(leer_base(ruta), anio))
    detalle = pd.concat(piezas, ignore_index=True)
    ds1 = detalle.rename(columns={"oficial_pct": "oficial_p", "privado_pct": "privado_p"})
    ds1 = ds1[["departamento", "año", "tecnica", "tecnologica", "universitaria", "oficial_p", "privado_p"]]
    ds1.columns = ["departamento", "año", "tecnica", "tecnologica", "universitaria", "oficial", "privado"]
    for c in ["tecnica", "tecnologica", "universitaria"]:
        ds1[c] = ds1[c].astype(int)
    return ds1.sort_values(["departamento", "año"]).reset_index(drop=True), detalle


## 2. Construcción del Dataset 1
Coloque los diez archivos Excel en la carpeta `snies_raw/` (el nombre debe contener el año). La lectura de los diez archivos tarda unos minutos.

In [ ]:
ds1, detalle = construir("snies_raw")
ds1.head(12)

Procesando snies_raw/matriculas_2016.xlsx
Procesando snies_raw/matriculas_2017.xlsx
Procesando snies_raw/matriculas_2018.xlsx
Procesando snies_raw/matriculas_2019.xlsx
Procesando snies_raw/matriculas_2020.xlsx
Procesando snies_raw/matriculas_2021.xlsx
Procesando snies_raw/matriculas_2022.xlsx
Procesando snies_raw/matriculas_2023.xlsx
Procesando snies_raw/matriculas_2024.xlsx
Procesando snies_raw/matriculas_2025.xlsx


,departamento,año,tecnica,tecnologica,universitaria,oficial,privado
0,Amazonas,2016,119,447,344,57.0,43.0
1,Amazonas,2017,7,318,315,82.5,17.5
2,Amazonas,2018,0,322,318,77.9,22.1
3,Amazonas,2019,5,412,319,89.7,10.3
4,Amazonas,2020,0,637,315,93.3,6.7
5,Amazonas,2021,23,329,315,97.1,2.9
6,Amazonas,2022,49,291,348,97.3,2.7
7,Amazonas,2023,30,256,382,99.9,0.1
8,Amazonas,2024,29,284,484,97.2,2.8
9,Amazonas,2025,50,291,540,97.7,2.3


## 3. Validaciones de calidad
### 3.1 Estructura, tipos, duplicados e integridad

In [ ]:
print("Dimensión:", ds1.shape, "(esperado: (330, 7))")
print("\nTipos de datos:\n", ds1.dtypes, sep="")

assert ds1.shape == (330, 7)
assert not ds1.duplicated(["departamento", "año"]).any(), "Hay duplicados departamento-año"
assert ds1["departamento"].nunique() == 33
assert sorted(ds1["año"].unique()) == list(range(2016, 2026))
assert (ds1.groupby("departamento").size() == 10).all(), "Algún territorio no tiene 10 años"
print("\n✔ Sin duplicados | 33 territorios | años 2016–2025 completos | 10 observaciones por territorio")

Dimensión: (330, 7) (esperado: (330, 7))

Tipos de datos:
departamento      object
año                int64
tecnica            int64
tecnologica        int64
universitaria      int64
oficial          float64
privado          float64
dtype: object

✔ Sin duplicados | 33 territorios | años 2016–2025 completos | 10 observaciones por territorio


### 3.2 Valores faltantes

In [ ]:
print(ds1.isna().sum())
assert ds1.isna().sum().sum() == 0
print("\n✔ No hay valores faltantes (NaN) en ninguna columna.")

departamento     0
año              0
tecnica          0
tecnologica      0
universitaria    0
oficial          0
privado          0
dtype: int64

✔ No hay valores faltantes (NaN) en ninguna columna.


### 3.3 Verificación `oficial + privado ≈ 100`

In [ ]:
suma = ds1["oficial"] + ds1["privado"]
print(suma.describe()[["min", "max"]])
assert ((suma - 100).abs() <= 0.15).all()
print("\n✔ En todas las filas oficial + privado = 100 (tolerancia por redondeo a 1 decimal: 0,15 pp).")

min    100.0
max    100.0
dtype: float64

✔ En todas las filas oficial + privado = 100 (tolerancia por redondeo a 1 decimal: 0,15 pp).


### 3.4 Verificación externa de totales
Se comparan los totales nacionales obtenidos (suma de los 33 territorios) con cifras publicadas por fuentes independientes:
- 2016: cifra del portal de consultas públicas del SNIES (captura de pantalla).
- 2017–2022: tabla *Matriculados por sector* del boletín estadístico de la Universidad Distrital (fuente MEN–SNIES).
- 2024 y 2025: matrícula del sector oficial según el informe *Matrícula en educación superior 2025* (MEN–SNIES).

In [ ]:
ref_total = {2016: 2394434, 2017: 2446314, 2018: 2440367, 2019: 2396250,
             2020: 2355603, 2021: 2448271, 2022: 2466228}
ref_oficial = {2016: 1194697, 2024: 1401639, 2025: 1545484}

tot = detalle.groupby("año")[["total", "oficial"]].sum().astype(int)
tot["ref_total"] = pd.Series(ref_total)
tot["dif_total"] = tot["total"] - tot["ref_total"]
tot["ref_oficial"] = pd.Series(ref_oficial)
tot["dif_oficial"] = tot["oficial"] - tot["ref_oficial"]
display(tot)
assert (tot["dif_total"].dropna() == 0).all() and (tot["dif_oficial"].dropna() == 0).all()
print("✔ Los totales coinciden exactamente con las fuentes de referencia disponibles.")

,total,oficial,ref_total,dif_total,ref_oficial,dif_oficial
año,,,,,,
2016,2394434,1194697,2394434.0,0.0,1194697.0,0.0
2017,2446314,1241790,2446314.0,0.0,NaN,NaN
2018,2440367,1242482,2440367.0,0.0,NaN,NaN
2019,2396250,1218130,2396250.0,0.0,NaN,NaN
2020,2355603,1203895,2355603.0,0.0,NaN,NaN
2021,2448271,1334174,2448271.0,0.0,NaN,NaN
2022,2466228,1341339,2466228.0,0.0,NaN,NaN
2023,2475833,1333971,NaN,NaN,NaN,NaN
2024,2553560,1401639,NaN,NaN,1401639.0,0.0


✔ Los totales coinciden exactamente con las fuentes de referencia disponibles.


**Control de 2016 por nivel de formación contra el portal:**

In [ ]:
niv16 = detalle[detalle["año"] == 2016][["tecnica", "tecnologica", "universitaria"]].sum().astype(int)
portal16 = pd.Series({"tecnica": 82585, "tecnologica": 638499, "universitaria": 1513201})
pd.DataFrame({"base consolidada": niv16, "portal": portal16, "diferencia": niv16 - portal16})

,base consolidada,portal,diferencia
tecnica,82585,82585,0
tecnologica,638412,638499,-87
universitaria,1513288,1513201,87


La técnica profesional coincide exactamente. Tecnológica y universitaria difieren en **±87 estudiantes (0,01 %)**; es una diferencia mínima, probablemente por una reclasificación entre niveles en la caja de datos del portal. **No se corrige** y se deja documentada.

### 3.5 Tratamiento de ceros y valores atípicos

In [ ]:
cols = ["tecnica", "tecnologica", "universitaria"]
ceros = ds1[(ds1[cols] == 0).any(axis=1)]
print(f"Filas con al menos un nivel en cero: {len(ceros)} de {len(ds1)}")
display(ceros.groupby("departamento").size().rename("filas con ceros").to_frame().T)
print("\nCeros por nivel:"); print((ds1[cols] == 0).sum())

Filas con al menos un nivel en cero: 60 de 330


departamento,Amazonas,Arauca,Caquetá,Cauca,Cesar,Chocó,Guainía,Guaviare,Nariño,Putumayo,Quindío,Vaupés,Vichada
filas con ceros,2,9,3,2,2,1,6,8,1,7,2,8,9



Ceros por nivel:
tecnica          60
tecnologica       0
universitaria     0
dtype: int64


**Decisión:** los ceros corresponden a territorios (principalmente de la Amazonía, Orinoquía y Pacífico) donde no hay matrícula en ese nivel de formación, es decir, son **ceros reales y no valores faltantes**.

### 3.6 Observación sobre el sector oficial
El SENA se contabiliza en el sector **oficial**, y su peso es grande en territorios con poca oferta universitaria. Por eso la participación oficial puede cambiar bruscamente de un año a otro. Ejemplos en el dataset:

In [ ]:
ds1[ds1["departamento"].isin(["Amazonas", "Bogotá, D.C."])].pivot(index="año", columns="departamento", values="oficial")

departamento,Amazonas,"Bogotá, D.C."
año,,
2016,57.0,29.6
2017,82.5,30.3
2018,77.9,29.8
2019,89.7,29.7
2020,93.3,30.7
2021,97.1,40.7
2022,97.3,39.6
2023,99.9,38.6
2024,97.2,37.8


Estos saltos son características de los datos y no errores de construcción. Se tendrán en cuenta al interpretar el Dataset 3 y el agrupamiento por sector oficial.

## 4. Exportación del Dataset 1

In [ ]:
ds1.to_csv("dataset1.csv", index=False)
print("Guardado dataset1.csv:", ds1.shape)
pd.read_csv("dataset1.csv").head()

Guardado dataset1.csv: (330, 7)


,departamento,año,tecnica,tecnologica,universitaria,oficial,privado
0,Amazonas,2016,119,447,344,57.0,43.0
1,Amazonas,2017,7,318,315,82.5,17.5
2,Amazonas,2018,0,322,318,77.9,22.1
3,Amazonas,2019,5,412,319,89.7,10.3
4,Amazonas,2020,0,637,315,93.3,6.7


## 5. Resumen de la Tarea 1
- Dataset 1 con **330 filas y 7 columnas** (33 territorios × 10 años), con los tipos solicitados (`str`, `int`, `float`).
- Sin duplicados, sin valores faltantes y con `oficial + privado = 100`.
- Totales nacionales verificados contra fuentes externas.
- Decisiones metodológicas y discrepancias documentadas (criterio de territorio, dato anual, ±87 en 2016, ceros reales, peso del SENA).